### Mugrade boilerplate

In [ ]:
### Run this cell to install and import the homework tests
%pip install --upgrade git+https://github.com/locuslab/mugrade.git
!wget -nc https://raw.githubusercontent.com/zkolter/llm_speedrun/refs/heads/main/part4_inference_tests.py

import mugrade
import os
from part4_inference_tests import *
os.environ["MUGRADE_HW"] = "Part 4 - Inference"
# Read the Mugrade API key from the environment or from a local .env file.
# Never hard-code the key. See .env.example in the repository root.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
if not os.environ.get("MUGRADE_KEY"):
    print("MUGRADE_KEY is not set. Cells that use @mugrade.submit_tests need it.")

### BPE

Paste your BPE code from Part 1 here.

In [2]:
### BEGIN YOUR CODE
import json
from collections import Counter
import re
from tqdm.auto import tqdm

class BPE:
    def __init__(self, filename=None):
        ### BEGIN YOUR CODE
        if filename is not None:
            with open(filename, "rt") as fobj:
                self.vocab, merge_keys, merge_values, self.special_tokens = json.load(fobj)
                self.merges = {
                    tuple(k): v for k, v in zip(merge_keys, merge_values)
                }
                self.special_tokens = {
                    int(k): v for k, v in self.special_tokens.items()
                }
        else:
            self.vocab = [chr(i) for i in range(256)]
            self.merges = {}
            self.special_tokens = {}
        ### END YOUR CODE

    @staticmethod
    def merge_pair(word, a, b, merged):
        ### BEGIN YOUR CODE
        i = 0
        while i < len(word) - 1:
            if word[i] == a and word[i+1] == b:
                word[i: i+2] = [merged]
            i+=1
        ### END YOUR CODE

    def replace_special_tokens(self, word):
        ### BEGIN YOUR CODE
        for k, v in self.special_tokens.items():
            word = word.replace(v, chr(k))
        return word
        ### END YOUR CODE


    def train(self, text, target_vocab_size):
        ### BEGIN YOUR CODE
        regex_matching = re.findall(r"\s\S+|\s+|\S+", text)
        frequency_counter = Counter(regex_matching)
        frequency_map = {
            self.replace_special_tokens(k): v for k, v in frequency_counter.items() if v > 1
        }

        # Build word_list_frequency_map without defaultdict
        word_list_frequency_map = {}
        
        for word, freq in frequency_map.items():
            filtered_word = tuple(
                ord(char) for char in word 
                if ord(char) not in self.special_tokens
            )
            
            if filtered_word:
                word_list_frequency_map[filtered_word] = \
                    word_list_frequency_map.get(filtered_word, 0) + freq

        words_as_lists, words_as_lists_frequencies = \
            [list(i) for i in word_list_frequency_map.keys()], list(word_list_frequency_map.values())


        for _ in range(len(self.vocab), target_vocab_size):
            existing_vocab_size = len(self.vocab)
            suitable_merge_pairs = Counter()

            for current_word_as_list, current_word_frequency in \
                zip(words_as_lists, words_as_lists_frequencies):

                for i in range(len(current_word_as_list) - 1):
                    suitable_merge_pairs[(current_word_as_list[i], current_word_as_list[i+1])] +=\
                        current_word_frequency

            # Break condition for testing
            # if len(suitable_merge_pairs) == 0:
            #     break

            symbol_a, symbol_b = suitable_merge_pairs.most_common()[0][0]
            self.vocab.append(self.vocab[symbol_a] + self.vocab[symbol_b])
            self.merges[(symbol_a, symbol_b)] = existing_vocab_size

            for word_as_list in words_as_lists:
                self.merge_pair(word_as_list, symbol_a, symbol_b, existing_vocab_size)

        if len(self.special_tokens) != 0:
            max_special_token = max(self.special_tokens.keys())
            self.vocab += [""] * (max_special_token - len(self.vocab) + 1)
            for k, v in self.special_tokens.items():
                self.vocab[k] = v
        ### END YOUR CODE

    def encode(self, text):
        ### BEGIN YOUR CODE
        tokens = []

        for word in re.findall(r"\s\S+|\s+|\S+", text):
            word_as_list = [ord(char) for char in self.replace_special_tokens(word)]
            # Special tokens are now single code points (e.g. chr(32760)). No merge
            # contains them, so they pass through as their own token IDs.
            
            if  len(word_as_list) == 0:
                continue

            while True:
                word_symbol_pairs_to_vocab_index_mapping = {}
                for i in range(len(word_as_list) - 1):
                    current_symbol_pair = \
                        (word_as_list[i], word_as_list[i+1])
                    if current_symbol_pair in self.merges:
                        word_symbol_pairs_to_vocab_index_mapping[current_symbol_pair] = \
                            self.merges[current_symbol_pair]

                if len(word_symbol_pairs_to_vocab_index_mapping) == 0:
                    break

                symbol_a, symbol_b = min_symbol_pair = min(
                    word_symbol_pairs_to_vocab_index_mapping,
                    key = word_symbol_pairs_to_vocab_index_mapping.get
                )
                self.merge_pair(word_as_list, symbol_a, symbol_b, 
                                word_symbol_pairs_to_vocab_index_mapping[min_symbol_pair])

            tokens += word_as_list

        return tokens
        ### END YOUR CODE

    def decode(self, tokens):
        ### BEGIN YOUR CODE
        return "".join([self.vocab[token] for token in tokens])
        ### END YOUR CODE

    def save(self, fname):
        ### BEGIN YOUR CODE
        with open(fname, "wt") as fobj:
            json.dump([
                self.vocab,
                list(self.merges.keys()),
                list(self.merges.values()),
                self.special_tokens
            ], fobj)
        ### END YOUR CODE
### END YOUR CODE

### LLM with efficient inference

Add KV caching and position-based RoPE to the LLM architecture from Part 2, then implement `generate()` to produce completions.

The new graded methods are `rope`, `multihead_attn`, `transformer_block`, `__call__`, and `generate`. Reuse the other functions from Part 2; they are not graded again here.

`generate` takes a batch of equally long prompts and returns a tensor containing each prompt followed by at most `max_tokens` new tokens. Sample from the last-token logits using the positive temperature `temp`. Pass an EOS token and stop each sequence when it generates that token. Padding after a sequence's first generated EOS is not graded. A zero token budget returns the prompts.


In [11]:
import torch
import math

def embedding(tokens, weight, dtype):
    ### BEGIN YOUR CODE
    return weight[tokens].to(dtype=dtype)
    ### END YOUR CODE

def linear(x, weight):
    ### BEGIN YOUR CODE
    return x @ weight.type_as(x)
    ### END YOUR CODE

def silu(x):
    ### BEGIN YOUR CODE
    return x / (1 + torch.exp(-x))
    ### END YOUR CODE

def rms_norm(x):
    ### BEGIN YOUR CODE
    x_typed = x.float()
    x_normed = x_typed / (
        (x_typed**2)
            .mean(
                dim = -1,
                keepdim = True,
            )
            .sqrt()
    )
    return x_normed.type_as(x)
    ### END YOUR CODE

def softmax(x):
    ### BEGIN YOUR CODE
    x_numerical_stable = \
        x - x.max(dim = -1, keepdim=True).values
    x_exp = x_numerical_stable.exp()
    return x_exp / (
        x_exp.sum(dim=-1, keepdim=True)
    )
    ### END YOUR CODE

def self_attn(q,k,v,mask):
    ### BEGIN YOUR CODE
    attention_score = softmax(
        q @ k.transpose(-2, -1) / math.sqrt(k.size(-1)) + mask
    )
    return attention_score @ v
    ### END YOUR CODE


class LLM:
    @mugrade.submit_tests
    def rope(self, x, pos=0):
        ### BEGIN YOUR CODE
        L = x.shape[-2]
        w1 = self.buffers["rope1"][pos:pos + L]
        w2 = self.buffers["rope2"][pos:pos + L]
        flipped = x.reshape(-1, 2).flip(-1).reshape_as(x)
        return (w1 * x + w2 * flipped).type_as(x)
        ### END YOUR CODE

    @mugrade.submit_tests
    def multihead_attn(self, x, layer, mask, pos=0, cache=None):
        ### BEGIN YOUR CODE
        B, L, D = x.shape
        q = linear(x, self.params[f"wq_{layer}"])
        k = linear(x, self.params[f"wk_{layer}"])
        v = linear(x, self.params[f"wv_{layer}"])
        wp = self.params[f"wp_{layer}"]
 
        q = q.reshape(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.reshape(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.reshape(B, L, self.num_heads, self.head_dim).transpose(1, 2)
 
        q = rms_norm(self.rope(q, pos))
        k = rms_norm(self.rope(k, pos))
 
        if cache is not None:
            c = cache[layer]
            c["k"][:, :, pos:pos + L] = k
            c["v"][:, :, pos:pos + L] = v
            k = c["k"][:, :, :pos + L]
            v = c["v"][:, :, :pos + L]
 
        attn_output = self_attn(q, k, v, mask)
 
        return linear(
            attn_output.transpose(1, 2).reshape(B, L, D),
            wp
        )
        ### END YOUR CODE

    def mlp(self, x, layer):
        ### BEGIN YOUR CODE
        return linear(
            silu(
                linear(
                    x,
                    self.params[f"w1_{layer}"]
                )
            ),
            self.params[f"w2_{layer}"]
        )
        ### END YOUR CODE

    @mugrade.submit_tests
    def transformer_block(self, x, layer, mask, pos=0, cache=None):
        ### BEGIN YOUR CODE
        x = x + \
            self.multihead_attn(
                rms_norm(x), layer, mask, pos=pos, cache=cache
            )
        return x + self.mlp(
            rms_norm(x), layer
        )
        ### END YOUR CODE

    def __init__(self, config):
        ### BEGIN YOUR CODE
        self.num_layers = config["depth"]
        self.dtype = config["dtype"]

        vocab_size = config["vocab_size"]
        sequence_len = config["seq_len"]

        hidden_dim = self.num_layers * config["aspect_ratio"]
        self.mlp_dim = hidden_dim * config["mlp_multiple"]

        self.head_dim = config["head_dim"]
        self.num_heads = hidden_dim // self.head_dim

        self.params = {}
        dim_scale, dim_scale_mlp = \
            math.sqrt(2/hidden_dim), math.sqrt(2/self.mlp_dim)

        self.params["embedding"] = torch.randn(
            vocab_size, hidden_dim
        ) * dim_scale
        self.params["output"] = torch.randn(
            hidden_dim, vocab_size
        ) * dim_scale

        for i in range(self.num_layers):
            self.params[f"wq_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"wk_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"wv_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"wp_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"w1_{i}"] = \
                torch.randn(
                    hidden_dim, self.mlp_dim
                ) * dim_scale
            self.params[f"w2_{i}"] = \
                torch.randn(
                    self.mlp_dim, hidden_dim
                ) * dim_scale_mlp

        self.buffers = {}
        mask = torch.full(
            (sequence_len, sequence_len),
            -float("inf"))
        self.buffers["mask"] = \
            torch.triu(mask, diagonal=1)


        freqs = config["rope_theta"] ** (
            (-2 * torch.arange(
                self.head_dim // 2
            ) / self.head_dim)
        )
        theta = torch.outer(
            torch.arange(sequence_len),
            freqs
        )

        cos, sin = torch.cos(theta), torch.sin(theta)
        self.buffers["rope1"] = \
            torch.stack([cos, cos], dim = -1).reshape(
                sequence_len, self.head_dim
            )
        self.buffers["rope2"] = \
            torch.stack([-sin, sin], dim = -1).reshape(
                sequence_len, self.head_dim
            )

        ### END YOUR CODE

    @mugrade.submit_tests
    def __call__(self, tokens, pos=0, cache=None):
        ### BEGIN YOUR CODE
        B, L = tokens.shape
        x = rms_norm(
            embedding(
                tokens,
                self.params["embedding"],
                self.dtype
            )
        )
 
        mask = self.buffers["mask"][pos:pos + L, :pos + L]
 
        for i in range(self.num_layers):
            x = self.transformer_block(
                x, i, mask, pos=pos, cache=cache
            )
 
        x_final = \
            rms_norm(x)
 
        return linear(
            x_final,
            self.params["output"]
        )
        ### END YOUR CODE

    @mugrade.submit_tests
    def generate(self, prompts, max_tokens=500, temp=0.7, eos=None):
        ### BEGIN YOUR CODE
        B, prompt_len = prompts.shape
        tokens = prompts.clone()
 
        if max_tokens == 0:
            return tokens
 
        cache = [
            {
                "k": torch.zeros(
                    B, self.num_heads, prompt_len + max_tokens, self.head_dim,
                    dtype=self.dtype, device=prompts.device
                ),
                "v": torch.zeros(
                    B, self.num_heads, prompt_len + max_tokens, self.head_dim,
                    dtype=self.dtype, device=prompts.device
                ),
            }
            for _ in range(self.num_layers)
        ]
 
        logits = self(tokens, pos=0, cache=cache)
        pos = prompt_len - 1
        done = torch.zeros(B, dtype=torch.bool, device=prompts.device)
 
        for step in range(max_tokens):
            probs = softmax(logits[:, -1, :].float() / temp)
            next_token = torch.multinomial(probs, 1).to(tokens.dtype)
            tokens = torch.cat([tokens, next_token], dim=1)
 
            if eos is not None:
                done = done | (next_token.squeeze(-1) == eos)
                if done.all():
                    break
 
            if step == max_tokens - 1:
                break
 
            pos += 1
            logits = self(next_token, pos=pos, cache=cache)
 
        return tokens
        ### END YOUR CODE

    def save(self, filename):
        ### BEGIN YOUR CODE
        torch.save(self.params, filename)
        ### END YOUR CODE

    def load(self, filename):
        ### BEGIN YOUR CODE
        self.params = \
            torch.load(filename, weights_only=True)
        ### END YOUR CODE

Mugrade: Submitting tests for function rope():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function multihead_attn():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function transformer_block():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function __call__():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function generate():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed


### Generating text

After you have implmeneted the functions above, you can use the following code to download the d30 model we demonstrated in class (you're also welcome to use your own d12 model if you've been able to train one, but the d30 uses the exact same architecture).  You should be able to use your new architecture to generate samples from the model.

In [ ]:
from huggingface_hub import hf_hub_download
import os
import json

repo = "zkolter/llm_speedrun"
filenames = ["llm.d30.pt", "config.d30.json", "tokenizer_50M.bpe"]

for filename in filenames:
    if not os.path.exists(filename):
        hf_hub_download(repo_id=repo, filename=filename, repo_type="dataset", local_dir=".")

In [ ]:
with open("config.d30.json", "rt") as f:
    config = json.load(f)
torch_types = {"float32": torch.float32, "bfloat16": torch.bfloat16}
config["dtype"] = torch_types[config["dtype"]]

llm = LLM(config)
llm.load("llm.d30.pt")
for k in llm.params: llm.params[k] = llm.params[k].cuda()
for k in llm.buffers: llm.buffers[k] = llm.buffers[k].cuda()
bpe = BPE(config["tokenizer"])

In [ ]:
prompt = "<DOCUMENT><USER>WHy is the sky blue?</USER><ASSISTANT>"
prompts = torch.tensor([bpe.encode(prompt)]).cuda()

eos = bpe.vocab.index("</ASSISTANT>")
tokens = llm.generate(prompts, max_tokens=500, temp=0.7, eos=eos)
print(bpe.decode(tokens.tolist()[0]))